# 02. Structured Ticket Extraction — via LLM Prompting with JSON Schema (Azure OpenAI Responses API)

This notebook builds directly on `01_named_entitities.py`: same Azure AI Foundry project, same model, same
ticket text and extraction goal — but this time the Responses API call includes a **strict JSON Schema**
(`text.format` with `type: "json_schema"`, `strict: True`). This forces the model's output to conform exactly
to the schema, so `json.loads(response.output_text)` can be trusted to succeed and produce predictable keys.

This is still **prompt-based LLM extraction**, not the Azure AI Language NER service — the schema constrains
the *shape* of the answer, not the underlying extraction quality/accuracy. It solves the "the model didn't
return valid JSON" reliability problem from notebook 01.

**Difficulty:** Beginner–Intermediate


## Prerequisites

**Python packages** (already covered by the repo's root `requirements.txt`):
- `openai`, `azure-ai-projects`, `azure-identity`, `python-dotenv` (same as notebook 01)
- `json` — Python standard library, no install needed

**Azure resources required:**
- An Azure AI Foundry project with a chat-capable model deployment that supports structured outputs
  (`json_schema` response format)

**Environment variables** (add to root `.env`):
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_MODEL_DEPLOYMENT=<your-deployment-name>
```

Auth is via `DefaultAzureCredential` (`az login`).


## What You'll Learn

- How to define a JSON Schema for LLM output (`type`, `properties`, `enum`, `required`, `additionalProperties`)
- How to pass that schema to the Responses API via `text.format` with `"strict": True`
- Why strict structured outputs eliminate the "invalid JSON" failure mode from prompt-only extraction
- How `enum` constrains categorical fields (e.g. entity `category`) to a fixed, known set of values
- How this compares to calling the dedicated Azure AI Language NER endpoint for entity extraction


### Step 1 — Connect to the Azure AI Foundry project (same setup as notebook 01)

Identical client setup: Entra ID auth via `DefaultAzureCredential`, `AIProjectClient` for the project, and
`get_openai_client()` for a standard OpenAI-compatible client.

💡 **Exam tip:** Reusing the same Foundry project/client code across multiple extraction tasks (NER, sentiment,
translation, structured extraction) reflects how, on AI-102, a single Azure AI Foundry project can back many
different "skills" — model deployments are the reusable building block, not one-resource-per-task.

🔄 **Alternatives:** None new here — see notebook 01 for the client-setup alternatives.


In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
deployment_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT"]

project = AIProjectClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

### Step 2 — Define the ticket text, prompt, and extraction JSON Schema

The `ticket_text` and `system_prompt` are the same as notebook 01. What's new is `extraction_schema`: a JSON
Schema object describing exactly what shape the response must take.

- `"entities"` is an array of objects, each requiring `"text"` (a string) and `"category"` (constrained to
  one of seven allowed values via `"enum"`).
- `"additionalProperties": False` at every object level means the model **cannot** add extra keys — this is
  what makes `strict` mode enforceable.
- `"required"` lists which keys must be present on every object.

💡 **Exam tip:** This maps conceptually to how Azure AI Language's prebuilt NER response is *always* shaped
the same way (a fixed set of fields: `text`, `category`, `subcategory`, `confidence_score`, offsets) regardless
of input — the Language service gives you that structural guarantee for free, without needing to hand-write a
schema, because it's a purpose-built API rather than a general LLM being steered by a schema.

🔄 **Alternatives:** Instead of hand-writing raw JSON Schema, you could use a Pydantic model and a helper
library (e.g. the OpenAI SDK's Pydantic-based structured output helpers) to define and validate the schema in
Python syntax instead of JSON syntax.


In [ ]:
ticket_text = """
Subject: VPN disconnect issue - escalation needed

Hi team, this is Sarah Chen from Acme Logistics writing in again about
ticket TKT-1042. Our Gold-tier SLA promises a 4 hour response time, and
we are now at hour 6 with no update. The VPN client keeps dropping every
10 minutes on our Windows fleet since the rollout of CloudXeus Connect
v3.2 last Tuesday. If this isn't resolved by end of day Friday we will
be requesting the $500 SLA breach credit outlined in our contract.
"""

system_prompt = """
You are a text analysis engine for CloudXeus support tickets.
Read the ticket text and respond with ONLY a JSON object containing:
- "entities": a list of objects, each with "text" and "category"
  (categories: person, organization, date, product, ticket_id, sla_tier, monetary_amount)
- "topics": a list of short topic labels describing what the ticket is about

Respond with JSON only. Do not include any other text.
"""

extraction_schema = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "The exact substring as it appears in the source text"
                    },
                    "category": {
                        "type": "string",
                        "enum": [
                            "person",
                            "organization",
                            "date",
                            "product",
                            "ticket_id",
                            "sla_tier",
                            "monetary_amount"
                        ]
                    }
                },
                "required": ["text", "category"],
                "additionalProperties": False
            }
        },
        "topics": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Short labels (2-4 words) describing what the text is about"
        }
    },
    "required": ["entities", "topics"],
    "additionalProperties": False
}

### Step 3 — Call the Responses API with `text.format` set to the schema

The `text` parameter's `format` sub-object is where structured outputs are configured:
- `"type": "json_schema"` selects schema-constrained output
- `"name"` is a label for the schema (useful for logging/debugging)
- `"schema"` is the JSON Schema itself
- `"strict": True` tells the model provider to enforce the schema at generation time (not just validate after
  the fact) — this is what gives you the reliability guarantee.

Because the output is now guaranteed valid JSON matching the schema, `json.loads(response.output_text)` is
safe to call directly, and we can confidently index into `result["entities"]` and `result["topics"]`.

💡 **Exam tip:** Structured/guaranteed JSON output is an Azure OpenAI (and OpenAI-compatible) model feature,
not something Azure AI Language exposes as a config option — Language's responses are already a fixed schema
by design because it's a dedicated NLP API, not a general text generator. Don't confuse "schema-constrained
LLM output" with the Language service's inherently structured API responses; they solve a similar-*looking*
problem for different underlying reasons.

🔄 **Alternatives:** Call the Azure AI Language `recognize_entities` prebuilt NER endpoint (notebook 07) when
you want guaranteed-structured entity extraction *without* needing to design a schema or depend on a large
general-purpose model — trading flexibility (custom categories) for lower latency/cost and a fixed, certified
category set.


In [ ]:
response = openai.responses.create(
    model=deployment_name,
    input=[
        {"type": "message", "role": "system", "content": system_prompt},
        {"type": "message", "role": "user", "content": ticket_text}
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "ticket_extraction",
            "schema": extraction_schema,
            "strict": True
        }
    }
)

result = json.loads(response.output_text)

print("=== Entities ===")
for entity in result["entities"]:
    print(f"  [{entity['category']}] {entity['text']} ")

print("\n=== Topics ===")
for topic in result["topics"]:
    print(f"  - {topic}")

## Summary

By adding a strict JSON Schema to the Responses API call, this notebook converted the free-form extraction
from notebook 01 into a reliable, directly-parseable structure. `result["entities"]` and `result["topics"]`
are safe to index without defensive parsing. The trade-off is upfront schema-design effort, and the schema
still can't guarantee *semantic* correctness (the model could still mis-categorize an entity) — only its
*shape*.


## Try It Yourself

1. **Easy:** Add a new category to the `enum` list (e.g. `"phone_number"`) and update the system prompt to
   match — rerun and see if the model picks it up.
2. **Intermediate:** Add a third top-level field to the schema, e.g. `"urgency": {"type": "string", "enum":
   ["low", "medium", "high"]}`, marked required, and update the prompt accordingly.
3. **Advanced:** Write a small validation function that cross-checks each extracted entity's `"text"` value
   actually appears verbatim in `ticket_text` (LLMs can still hallucinate values that satisfy the schema type
   but aren't a real substring) and reports any mismatches.
